### 5. Создание внешней таблицы (EXTERNAL TABLE) в Hive поверх существующих Parquet-файлов:

Название: ebay_raw_parquet

Убедимся, что схема таблицы полностью соответствует метаданным файлов.


---

Узнаем схему данных в Parquet-файлах

In [ ]:
# import pyspark as spark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Hadoop_Assignment_VSCode") \
    .enableHiveSupport() \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/10 12:11:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:

parquet_file_path = "/user/s.yudin/ebay/snapshot_dt=2026-02-28"

# Читаем файл(ы) в Spark DataFrame
df_raw = spark.read.parquet(parquet_file_path)

# Выводим схему таблицы на экран
print("Схема Parquet-файла:")
df_raw.printSchema()

# Также можно посмотреть первые несколько строк данных для наглядности
print("\nПример данных:")
df_raw.show(5, truncate=False)


Схема Parquet-файла:
root
 |-- snapshot_dt: date (nullable = true)
 |-- itemid: string (nullable = true)
 |-- title: string (nullable = true)
 |-- price: double (nullable = true)
 |-- top_rated_buying_experience: boolean (nullable = true)
 |-- search_category: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- sub_category_id: string (nullable = true)
 |-- sub_category_name: string (nullable = true)
 |-- sub_sub_category_id: string (nullable = true)
 |-- sub_sub_category_name: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- item_condition: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- seller_feedback_percentage: double (nullable = true)
 |-- seller_feedback_score: long (nullable = true)
 |-- location_country: string (nullable = true)
 |-- shipping_cost_type: string (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- shipping_currency: string (nullable = tr

+-----------+------------+--------------------------------------------------------------------------------+-----+---------------------------+---------------+-----------+-----------------------------+---------------+-------------------------+-------------------+-------------------------+--------+--------------+-------------+--------------------------+---------------------+----------------+------------------+-------------+-----------------+-----------------------+------------------+-------------------+
|snapshot_dt|itemid      |title                                                                           |price|top_rated_buying_experience|search_category|category_id|category_name                |sub_category_id|sub_category_name        |sub_sub_category_id|sub_sub_category_name    |currency|item_condition|seller_name  |seller_feedback_percentage|seller_feedback_score|location_country|shipping_cost_type|shipping_cost|shipping_currency|estimated_delivery_days|buying_options    |item_crea

Создадим запрос в веб интерфесе Hue

---


```sql
create database if not exists s_yudin; 
use s_yudin;

CREATE EXTERNAL TABLE IF NOT EXISTS s_yudin.ebay_raw_parquet (
        snapshot_dt DATE,
        itemid STRING,
        title STRING,
        price DOUBLE,
        top_rated_buying_experience BOOLEAN,
        search_category STRING,
        category_id STRING,
        category_name STRING,
        sub_category_id STRING,
        sub_category_name STRING,
        sub_sub_category_id STRING,
        sub_sub_category_name STRING,
        currency STRING,
        item_condition STRING,
        seller_name STRING,
        seller_feedback_percentage DOUBLE,
        seller_feedback_score BIGINT,
        location_country STRING,
        shipping_cost_type STRING,
        shipping_cost DOUBLE,
        shipping_currency STRING,
        estimated_delivery_days BIGINT,
        buying_options STRING,
        item_creation_ts TIMESTAMP
)
STORED AS PARQUET
LOCATION '/user/s.yudin/ebay';

SELECT * FROM s_yudin.ebay_raw_parquet LIMIT 5;
```
---

Результат выпонения и проверки соответствия схеме данных:

![](./../img/ex5.png)

### 6. Создайте целевую таблицу, настроив партиционирование по дате выгрузки (event_date или аналогичному полю):
Название: ebay_listings_optimized

---


```sql
use s_yudin;

CREATE TABLE IF NOT EXISTS s_yudin.ebay_listings_optimized (
        itemid STRING,
        title STRING,
        price DOUBLE,
        top_rated_buying_experience BOOLEAN,
        search_category STRING,
        category_id STRING,
        category_name STRING,
        sub_category_id STRING,
        sub_category_name STRING,
        sub_sub_category_id STRING,
        sub_sub_category_name STRING,
        currency STRING,
        item_condition STRING,
        seller_name STRING,
        seller_feedback_percentage DOUBLE,
        seller_feedback_score BIGINT,
        location_country STRING,
        shipping_cost_type STRING,
        shipping_cost DOUBLE,
        shipping_currency STRING,
        estimated_delivery_days BIGINT,
        buying_options STRING,
        item_creation_ts TIMESTAMP
    )
    PARTITIONED BY (snapshot_dt DATE)
    STORED AS PARQUET;
```

---

### 7. Перенос данных (ETL Process)
Перенести данные из «сырой» текстовой таблицы в оптимизированную партиционированную

---

Проверим настройки командой SET (hive.exec.dynamic.partition=true - OK; hive.exec.dynamic.partition.mode=nonstrict - OK;)

Перенесем данные:

```sql
INSERT OVERWRITE TABLE s_yudin.ebay_listings_optimized PARTITION (snapshot_dt)
SELECT
    itemid,
    title,
    price,
    top_rated_buying_experience,
    search_category,
    category_id,
    category_name,
    sub_category_id,
    sub_category_name,
    sub_sub_category_id,
    sub_sub_category_name,
    currency,
    item_condition,
    seller_name,
    seller_feedback_percentage,
    seller_feedback_score,
    location_country,
    shipping_cost_type,
    shipping_cost,
    shipping_currency,
    estimated_delivery_days,
    buying_options,
    item_creation_ts,
    snapshot_dt
FROM s_yudin.ebay_raw_parquet
WHERE snapshot_dt IS NOT NULL;
```

Проверим результат

```sql
SELECT snapshot_dt, COUNT(*) FROM s_yudin.ebay_listings_optimized GROUP BY snapshot_dt ORDER BY snapshot_dt;
```

![](./../img/ex7.png)
